# CG visualization with PyVis

In [1]:
from pyvis.network import Network
import networkx as nx

# Create a PyVis Network object
# The 'layout' parameter enables hierarchical positioning
net = Network(height="800px", width="100%", notebook=True, directed=True, layout=True)

In [2]:
import random
from pyvis.network import Network

net = Network(height="800px", width="100%", notebook=True, directed=True)

# --- 1. Define Horizontal Zones (Y-coordinates) for Each Level ---
# This creates the distinct horizontal parts you described.
level_y_coords = {
    0: 600,  # Bottom zone for words
    1: 350,  # Middle zone for L1 concepts
    2: 100,  # Top zone for L2 concept
    3: -50   # Final prediction
}

# --- 2. Add Nodes with Locked Vertical Positions ---
# We set 'y' and 'fixed=True' for each node.
# The physics engine can now only move them horizontally.

# Level 0: Original words
level_0_nodes = ["I", "prefer", "the", "morning", "flight", "through", "Denver"]
for i, node_name in enumerate(level_0_nodes):
    net.add_node(
        node_name,
        label=node_name,
        level=0,
        y=int(level_y_coords[0] * (1+0.1*random.random())),
        fixed={'y': True}, # Lock vertical position
        shape='ellipse',
        color='#D3D3D3',
        font={'color': 'black'}
    )

# Level 1: Abstract concepts
level_1_concepts = {
    "A": "Flight Preference",
    "B": "Morning Departure",
    "C": "Route via Denver"
}
colors_l1_nodes = {"A": '#a3c4f3', "B": '#fde4a3', "C": '#f4a2a1'}
for node_id, node_label in level_1_concepts.items():
    net.add_node(
        node_id,
        label=node_label,
        level=1,
        y=int(level_y_coords[1] * (1+0.1*random.random())),
        fixed={'y': True}, # Lock vertical position
        shape='ellipse',
        color=colors_l1_nodes[node_id]
    )

# Level 2: Higher-level concept
net.add_node(
    "G",
    label="Travel Plan Details",
    level=2,
    y=int(level_y_coords[2] * (1+0.1*random.random())),
    fixed={'y': True}, # Lock vertical position
    shape='ellipse',
    color='#b39ddb'
)

# Prediction Node
net.add_node(
    "Prediction",
    label="Prediction",
    level=3,
    y=level_y_coords[3],
    shape='box',
    color='#ff6f61',
    fixed=True
)


# --- 3. Add Edges ---
# Intra-level edges remain as curved arcs
edge_smooth_options = {'type': 'curvedCW', 'roundness': 0.15}

# Edges within Level 0
net.add_edge("I", "prefer", dashes=False, color='gray', smooth=edge_smooth_options)
net.add_edge("prefer", "flight", dashes=False, color='gray', smooth=edge_smooth_options)
net.add_edge("the", "flight", dashes=False, color='gray', smooth=edge_smooth_options)
net.add_edge("morning", "flight", dashes=False, color='gray', smooth=edge_smooth_options)
net.add_edge("flight", "through", dashes=False, color='gray', smooth=edge_smooth_options)
net.add_edge("through", "Denver", dashes=False, color='gray', smooth=edge_smooth_options)

# Edges within Level 1
net.add_edge("A", "B", dashes=False, color='black', smooth=edge_smooth_options)
net.add_edge("B", "C", dashes=False, color='black', smooth=edge_smooth_options)

# Assignment edges between levels
assignment_edges_l0_l1 = {
    "A": ["I", "prefer", "the", "flight"],
    "B": ["morning", "flight"],
    "C": ["flight", "through", "Denver"]
}
colors_l1_edges = {"A": "rgba(163, 196, 243, 0.5)", "B": "rgba(253, 228, 163, 0.5)", "C": "rgba(244, 162, 161, 0.5)"}
for concept, words in assignment_edges_l0_l1.items():
    for word in words:
        net.add_edge(word, concept, dashes=True, color=colors_l1_edges[concept], arrows='')

net.add_edge("A", "G", dashes=True, color="rgba(179, 157, 219, 0.5)", arrows='')
net.add_edge("B", "G", dashes=True, color="rgba(179, 157, 219, 0.5)", arrows='')
net.add_edge("C", "G", dashes=True, color="rgba(179, 157, 219, 0.5)", arrows='')
net.add_edge("G", "Prediction", dashes=True, color="rgba(255, 111, 97, 0.7)")


# --- 4. Configure Physics for Horizontal Spreading ---
# The physics engine will now only affect the x-coordinates,
# creating the desired spreading effect within each locked zone.
options = """
{
  "physics": {
    "barnesHut": {
      "gravitationalConstant": -1000,
      "centralGravity": 0.1,
      "springLength": 150,
      "avoidOverlap": 0.8
    },
    "minVelocity": 0.0
  }
}
"""

net.set_options(options)

# Generate the final graph
net.show("zoned_layout_graph.html")

zoned_layout_graph.html


In [3]:
import random
import networkx as nx
from pyvis.network import Network
from typing import List, Tuple, Any, Dict

class HierarchicalGraphVisualizer:
    """
    Generates an interactive, multi-layered network graph from NetworkX graphs.

    This class creates a visualization with distinct horizontal "zones" for each level,
    where nodes can spread out freely. It's designed to show how lower-level
    nodes (e.g., tokens) are grouped into higher-level abstract concepts.
    """

    # --- Configuration Constants ---
    LEVEL_Y_COORDS = {0: 600, 1: 350, 2: 100, 3: -50}
    NODE_COLORS = {
        0: '#D3D3D3',
        1: '#a3c4f3',
        2: '#b39ddb',
        'prediction': '#ff6f61'
    }
    PHYSICS_OPTIONS = """
    {
      "physics": {
        "barnesHut": {
          "gravitationalConstant": -10000,
          "centralGravity": 0.1,
          "springLength": 150,
          "avoidOverlap": 0.5
        },
        "minVelocity": 0.75
      }
    }
    """

    def __init__(self,
                 g0: nx.Graph,
                 g1: nx.Graph,
                 assignments: List[Tuple[Any, Any, float]],
                 g0_names: Dict[Any, str] = None,
                 g1_names: Dict[Any, str] = None):
        """
        Initializes the visualizer with graph data.

        Args:
            g0 (nx.Graph): A NetworkX graph for Level 0.
            g1 (nx.Graph): A NetworkX graph for Level 1.
            assignments (List[Tuple[Any, Any, float]]): A list of tuples, where each
                tuple represents an assignment `(node_from_g0, node_from_g1, weight)`.
            g0_names (Dict[Any, str], optional): A dictionary mapping node IDs in g0
                to their display names. If None, the node IDs themselves are used.
            g1_names (Dict[Any, str], optional): A dictionary mapping node IDs in g1
                to their display names. If None, the node IDs themselves are used.
        """
        self.g0 = g0
        self.g1 = g1
        self.assignments = assignments
        self.g0_names = g0_names or {n: str(n) for n in g0.nodes()}
        self.g1_names = g1_names or {n: str(n) for n in g1.nodes()}
        self.net = Network(height="800px", width="100%", notebook=True, directed=True)

    def _add_nodes(self):
        """Adds and styles all nodes for all levels to the PyVis graph."""
        # Level 0 Nodes
        for node in self.g0.nodes():
            self.net.add_node(
                node,
                label=self.g0_names.get(node, str(node)),
                level=0,
                y=int(self.LEVEL_Y_COORDS[0] * (1 + 0.2 * random.random())),
                fixed={'y': True},
                shape='ellipse',
                color=self.NODE_COLORS[0],
                font={'color': 'black'}
            )

        # Level 1 Nodes
        for node in self.g1.nodes():
            self.net.add_node(
                node,
                label=self.g1_names.get(node, str(node)),
                level=1,
                y=int(self.LEVEL_Y_COORDS[1] * (1 + 0.2 * random.random())),
                fixed={'y': True},
                shape='ellipse',
                color=self.NODE_COLORS[1]
            )

        # Higher-level and Prediction Nodes
        g_node_id = "L2_Aggregate"
        self.net.add_node(
            g_node_id,
            label="L1 Concepts",
            level=2,
            y=self.LEVEL_Y_COORDS[2],
            fixed={'y': True},
            shape='ellipse',
            color=self.NODE_COLORS[2]
        )

        pred_node_id = "Prediction"
        self.net.add_node(
            pred_node_id,
            label="Prediction",
            level=3,
            x=0,
            y=self.LEVEL_Y_COORDS[3],
            fixed=True,
            shape='box',
            color=self.NODE_COLORS['prediction']
        )


    def _add_edges(self):
        """Adds and styles all edges (intra-level and inter-level)."""
        smooth_options = {'type': 'curvedCW', 'roundness': 0.15}

        # Intra-level edges for G0
        for u, v in self.g0.edges():
            self.net.add_edge(u, v, dashes=False, color='gray', smooth=smooth_options)

        # Intra-level edges for G1
        for u, v in self.g1.edges():
            self.net.add_edge(u, v, dashes=False, color='black', smooth=smooth_options)

        # --- MODIFICATION ---
        # Inter-level assignment edges (L0 -> L1) now reflect their weight.
        for u, v, w in self.assignments:
            # Assume w is a float between 0.0 and 1.0

            # 1. Higher weight = more opaque (less transparent)
            alpha = max(0.15, w) # Set minimum visibility
            color = f'rgba(180, 180, 180, {alpha})'

            # 2. Higher weight = thicker line
            width = 0.5 + (w * 2)

            # 3. Higher weight = more solid dash pattern (longer line, shorter gap)
            dash_length = int(7)
            gap_length = int(4)
            dashes = [dash_length, gap_length]

            self.net.add_edge(
                u, v,
                dashes=dashes,
                color=color,
                width=width,
                arrows='',
                title=f"Weight: {w:.2f}" # Hover-over title
            )
        # --- END MODIFICATION ---

        # Edges from L1 to L2 aggregate node
        g_node_id = "L2_Aggregate"
        for node in self.g1.nodes():
            self.net.add_edge(node, g_node_id, das=True, color='rgba(179, 157, 219, 0.5)', arrows='')

        # Final edge to prediction
        self.net.add_edge(g_node_id, "Prediction", dashes=True, color='rgba(255, 111, 97, 0.7)')


    def generate_html(self, filename: str = "hierarchical_graph.html"):
        """
        Generates and saves the interactive HTML graph file.

        Args:
            filename (str): The name of the output HTML file.
        """
        self._add_nodes()
        self._add_edges()
        self.net.set_options(self.PHYSICS_OPTIONS)
        self.net.show(filename)
        print(f"Graph saved to {filename}")

In [4]:

# --- Example Usage ---
if __name__ == '__main__':
    # 1. Create sample NetworkX graphs for Level 0 and Level 1
    g0 = nx.Graph()
    g0.add_nodes_from(["I", "prefer", "the", "morning", "flight", "through", "Denver"])
    g0.add_edges_from([("I", "prefer"), ("prefer", "flight"), ("the", "flight"),
                       ("morning", "flight"), ("flight", "through"), ("through", "Denver")])

    g1 = nx.Graph()
    g1.add_nodes_from(["A", "B", "C"])
    g1.add_edges_from([("A", "B"), ("B", "C")])

    # Define display names for the abstract nodes in Level 1
    level_1_names = {
        "A": "Flight Preference",
        "B": "Morning Departure",
        "C": "Route via Denver"
    }

    # 2. Define the assignments between L0 and L1 with weights
    assignments_data = [
        ("I", "A", 0.8), ("prefer", "A", 0.9), ("the", "A", 0.5),
        ("flight", "A", 1.0), ("flight", "B", 1.0), ("flight", "C", 0.9),
        ("morning", "B", 0.9),
        ("through", "C", 0.8), ("Denver", "C", 1.0)
    ]

    # 3. Instantiate the visualizer with the data
    visualizer = HierarchicalGraphVisualizer(
        g0=g0,
        g1=g1,
        assignments=assignments_data,
        g1_names=level_1_names
    )

    # 4. Generate the HTML file
    visualizer.generate_html("final_graph_from_class.html")


final_graph_from_class.html
Graph saved to final_graph_from_class.html


In [5]:
import os

os.chdir("/media/trdp/STORAGE/3.Trabalho/PhD-Research/ThesisExperiments/3.ToValidate/TextualHGNN")

In [6]:

import json
from src.utils.general_utils import load_config
import torch
import logging

LANG = "portuguese_voto"
DATASET = "STF_HC_Voto_Relatorio"
CONFIG = load_config(LANG, "src/utils/config.json")
ROOT = f"data/datasets/{DATASET}"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH = "models/grid_search/STF_HC_Voto_Relatorio/best_model_20250829_234323_lk0.0_en0.0_rc0.0_ct0.0_bl1.0_rp0.0_l20.0.pth"


logging.info(f"============  STARTING CG EXPERIMENT {DATASET}  ============")
logging.info(f"CONFIG: \n{json.dumps(CONFIG, indent=3)}\n")
logging.info(f"MODEL CHECKPOINT: {MODEL_PATH}")

weights = torch.load(MODEL_PATH)
EMBEDDING_PATH = "data/external/embeddings/glove_legal_100.bin"
EMBEDDINGS_DATABASE_PATH = f"data/oracle/embeddings_{DATASET}.db"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = f"data/datasets/{DATASET}"
DOCUMENTS_TO_EXPLAIN = 1


/tmp/ipykernel_46537/1244732077.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weights = torch.load(MODEL_PATH)


In [7]:

weights_best_model = torch.load(MODEL_PATH)

/tmp/ipykernel_46537/1807869396.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weights_best_model = torch.load(MODEL_PATH)


In [8]:
from src.models.graph_classification.gnn import DiffPool

best_diffpool_model = DiffPool(
    max_num_nodes=3000,
    in_channels=100,
    hidden_channels=100,
    out_channels=2,
    inner_channels=32,
    softmax_assign=True,
    decrease_proportion=0.1,
)
best_diffpool_model.load_state_dict(weights_best_model)
best_diffpool_model = best_diffpool_model.to(device=DEVICE)

RuntimeError: Error(s) in loading state_dict for DiffPool:
	size mismatch for lin1.weight: copying a param with shape torch.Size([100, 100]) from checkpoint, the shape in current model is torch.Size([32, 100]).
	size mismatch for lin1.bias: copying a param with shape torch.Size([100]) from checkpoint, the shape in current model is torch.Size([32]).
	size mismatch for lin2.weight: copying a param with shape torch.Size([2, 100]) from checkpoint, the shape in current model is torch.Size([2, 32]).

In [ ]:
from pathlib import Path
from src.data.text_graph_dataset_ondisk import TextGraphDatasetOnDisk
import torch_geometric.transforms as T

tgd_test = TextGraphDatasetOnDisk(
    root=ROOT,
    split="test",
    batch_size=1,
    node_feature_size=100,
    transform=T.ToDense(num_nodes=1000),
    max_num_nodes=1000,
    lang=LANG
)

nx_graphs = Path(ROOT) / "test" / "interim"
nx_graphs = str(nx_graphs) + "/*.pt"

In [ ]:
data_sample = tgd_test[0].to(DEVICE)

In [ ]:
data_sample_id = int.from_bytes(data_sample.doc_id, byteorder='little')

with torch.no_grad():
    prediction = best_diffpool_model(data_sample.x, data_sample.adj, debug=True)

In [18]:
import torch
x = torch.randn(5, 100)*0.01
x[3:] = 1e-10
x

tensor([[ 2.4399e-03, -5.7793e-03, -6.2379e-03,  8.2469e-03, -1.4206e-03,
         -8.5586e-03, -5.5593e-03, -1.9756e-02,  5.0059e-03,  5.1832e-04,
         -3.0091e-03,  2.4562e-03,  7.1600e-03, -8.7976e-03, -1.1991e-02,
          8.8638e-03, -1.0946e-02, -1.3301e-03,  8.0573e-03, -2.8767e-03,
          1.6952e-03, -2.4553e-03, -1.9023e-02, -1.9176e-02, -2.0016e-03,
          7.1466e-03, -2.0219e-02,  3.5270e-03,  1.0268e-02, -3.2599e-03,
         -1.1671e-02,  7.3766e-03, -1.0959e-02, -1.4639e-02, -1.0503e-02,
         -8.5444e-03,  4.9782e-03, -4.3014e-03,  5.9591e-04,  2.0878e-03,
         -3.0188e-03,  7.6435e-03,  1.1739e-02,  5.8875e-03, -2.2347e-02,
          8.3117e-03,  1.5729e-06, -1.8720e-03,  2.8517e-03, -1.0597e-02,
          2.4238e-02, -9.3837e-03,  1.9828e-03,  1.1258e-04,  1.3116e-02,
          9.3693e-03,  1.2002e-02, -2.0375e-03, -2.2578e-03,  8.6034e-03,
          1.7933e-02,  5.7174e-03,  9.5053e-03, -8.0605e-03, -2.7691e-03,
         -4.0130e-03,  1.2937e-02,  1.

In [23]:
# (x_l0.abs().sum(dim=1) > 1e-5)
(x.abs().sum(dim=1) > 1e-5).nonzero().max().item() + 1

3